In [ ]:
# run this notebook after 23_generate_family_files_R
# use this notebook to identify missense and lof variants in dna repair genes and identify non-reference 
# genotypes in trio parents and sibling pairs at these loci
# make sure you have a list of dna repair genes (./repair_genes_map_final.txt)
# after this, run 25_missense_lof_R

In [ ]:
import os
import pandas as pd
bucket=os.getenv("WORKSPACE_BUCKET")
import hail as hl
hl.init(default_reference="GRCh38", tmp_dir = f"{bucket}/tmp")

In [ ]:
genes = pd.read_csv("./repair_genes_map_final.txt", sep = " ", header=None)
genelist = genes.iloc[:,1].tolist()

In [ ]:
auxiliary_path = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux"
vat_path = f'{auxiliary_path}/vat/vat_complete.bgz.tsv.gz'
vat_table = hl.import_table(vat_path, force=True, quote='"', delimiter="\t", force_bgz=True)

In [ ]:
genes = hl.literal(set(genelist))  # set => o(1) contains; broadcast to workers
ht = vat_table.filter(genes.contains(vat_table.gene_id))
ht = ht.filter(ht.is_canonical_transcript == "true")
ht = ht.filter(ht.consequence.contains("missense_variant") | 
              ht.consequence.contains("transcript_ablation") |
              ht.consequence.contains("splice_acceptor_variant") |
              ht.consequence.contains("splice_donor_variant") |
              ht.consequence.contains("stop_gained") |
              ht.consequence.contains("frameshift_variant"))
ht.export(f"{bucket}/vat/vat_repair_genes_deleterious_missense_lof_final.tsv.bgz")

In [ ]:
trio = pd.read_csv(f"{bucket}/relatedness/trios_final.txt", sep = "\t")
parents = trio.loc[:,"par1"].astype(str).tolist() + trio.loc[:,"par2"].astype(str).tolist() 

In [ ]:
sib = pd.read_csv(f"{bucket}/relatedness/sibs_final.txt", sep = "\t")
sibs = sib.loc[:,"idv1"].astype(str).tolist() + sib.loc[:,"idv2"].astype(str).tolist()
par_sibs = parents + sibs

In [ ]:
ht = hl.import_table(f"{bucket}/vat/vat_repair_genes_deleterious_missense_lof_final.tsv.bgz")
mt = hl.read_matrix_table(f"{bucket}/data/twins_dups_sibs_trios_rep.mt")
ht_keys = (ht.select(
              locus   = hl.locus(ht.contig, hl.int32(ht.position), reference_genome="GRCh38"),
              alleles = hl.array([ht.ref_allele, ht.alt_allele])
          )
          .key_by('locus', 'alleles'))
ht.count()

In [ ]:
mt = mt.filter_rows(hl.is_defined(ht_keys[mt.row_key]))
mt = mt.filter_cols(hl.literal(par_sibs).contains(mt.s))
# filter to gq >= 30 
mt = mt.annotate_entries(GT = hl.or_missing(mt.GQ >= 30, mt.GT))
# filter to ft passing
mt = mt.annotate_entries(
    GT = hl.if_else(mt.FT == "FAIL", hl.missing(mt.GT.dtype), mt.GT, missing_false=True)
)
# keep anything non-ref
mt = mt.filter_rows(hl.agg.any(mt.GT.is_non_ref()))
mt.GT.export(f"{bucket}/vat/vat_repair_genes_missense_lof_trio_parents_siblings_nonref_GT.tsv.bgz")

In [ ]:
%%bash 

gsutil cp $WORKSPACE_BUCKET/vat/vat_repair_genes_deleterious_missense_lof_final.tsv.bgz .
gsutil cp $WORKSPACE_BUCKET/vat/vat_repair_genes_missense_lof_trio_parents_siblings_nonref_GT.tsv.bgz . 
mv vat_repair_genes_missense_lof_trio_parents_siblings_nonref_GT.tsv.bgz vat_repair_genes_missense_lof_trio_parents_siblings_nonref_GT.tsv.gz
gunzip vat_repair_genes_missense_lof_trio_parents_siblings_nonref_GT.tsv.gz 
mv vat_repair_genes_deleterious_missense_lof_final.tsv.bgz vat_repair_genes_deleterious_missense_lof_final.tsv.gz
gunzip vat_repair_genes_deleterious_missense_lof_final.tsv.gz 